In [1]:
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


In [4]:
from pyspark.sql import SparkSession

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("TugasMandiri4-PengenalanPySpark") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

26/09/16 23:02:52 WARN Utils: Your hostname, krock-PCPartner resolves to a loopback address: 127.0.1.1; using 10.194.253.236 instead (on interface wlp1s0)
26/09/16 23:02:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 23:02:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/16 23:02:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/16 23:02:59 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


SparkSession berhasil dibuat!
Versi Spark: 3.5.9


In [6]:
# A. Membaca dan Eksplorasi Awal 

# Membaca data langsung dari HDFS
path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv"
df_tugas4 = spark.read.csv(path_hdfs, header=True, inferSchema=True)

# Menampilkan skema
df_tugas4.printSchema()

# Menampilkan jumlah baris
print("Jumlah baris:", df_tugas4.count())

# Menampilkan 10 baris pertama
df_tugas4.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)



Jumlah baris: 1000


+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|
|ORD-3004|2026-09-10 00:00:00|        Rumah Tangga|Yogyakarta|          10|       60000|         E-Wallet|   4.0|
|ORD-3005|2026-09-09 00:00:00|             Fashion| Purworejo|           5|       20000|

In [11]:
# B. Menangani Data Kosong
from pyspark.sql.functions import col, avg

# 1. Cek jumlah nilai null pada kolom rating
jumlah_null = df_tugas4.filter(col("rating").isNull()).count()
print(f"Jumlah baris dengan rating kosong (null): {jumlah_null}")

# 2. Mengisi nilai kosong (imputasi) menggunakan rata-rata rating
rata_rating = df_tugas4.select(avg("rating")).first()[0]
df_clean = df_tugas4.na.fill({"rating": rata_rating})

# Verifikasi
df_clean.show(5)


# Menggunakan df.na.fill() dengan mengisi nilai kosong menggunakan nilai rata-rata daripada menghapus baris (df.na.drop()) 
# agar tidak kehilangan data transaksi penting lainnya yang berada di baris yang sama hanya karena kolom rating kosong.

Jumlah baris dengan rating kosong (null): 204
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|
|ORD-3004|2026-09-10 00:00:00|        Rumah Tangga|Yogyakarta|          10|       60000|         E-Wallet|   4.0|
+--------+-------------------+------------

In [13]:
# C. Transformasi Data
from pyspark.sql.functions import col, when

df_transformed = df_clean.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan")) \
                         .withColumn("tier_transaksi", when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))

df_transformed.show(5)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

In [27]:
# D. Analisis dengan GroupBy
from pyspark.sql.functions import col, sum, count, avg

# 1. Kategori yang memiliki pendapatan tertinggi 
print("1. Kategori yang memiliki pendapatan tertinggi: ")
df_transformed.groupBy("kategori") \
    .agg(sum("total_pendapatan").alias("total_pendapatan")) \
    .orderBy(col("total_pendapatan").desc()) \
    .show(1)

# 2. Kota mana yang memiliki jumlah transaksi tier "Besar" terbanyak?
print("\n2. Kota yang memiliki jumlah transaksi tier 'Besar' terbanyak: ")
df_transformed.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .agg(count("order_id").alias("jumlah_transaksi_besar")) \
    .orderBy(col("jumlah_transaksi_besar").desc()) \
    .show(1)

# 3. Rata-rata Rating per Metode Pembayaran
print("\n3. Rata-rata Rating per Metode Pembayaran: ")
df_transformed.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rata_rating")) \
    .orderBy(col("rata_rata_rating").desc()) \
    .show()

1. Kategori yang memiliki pendapatan tertinggi: 


+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row


2. Kota yang memiliki jumlah transaksi tier 'Besar' terbanyak: 
+----+----------------------+
|kota|jumlah_transaksi_besar|
+----+----------------------+
|Solo|                    92|
+----+----------------------+
only showing top 1 row


3. Rata-rata Rating per Metode Pembayaran: 
+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD| 4.167310656870009|
|    Transfer Bank|   4.1592349097265|
|         E-Wallet| 4.137728643216084|
|     Kartu Kredit|4.1179474608816475|
+-----------------+------------------+



In [28]:
# E. Menyimpan Hasil ke HDFS
output_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_olahan_september"
df_transformed.write.mode("overwrite").csv(output_hdfs, header=True)

!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_olahan_september

Found 2 items
-rw-r--r--   3 krock supergroup          0 2026-09-16 23:24 /user/mahasiswa/tugas4/hasil_olahan_september/_SUCCESS
-rw-r--r--   3 krock supergroup     100356 2026-09-16 23:24 /user/mahasiswa/tugas4/hasil_olahan_september/part-00000-c5803b58-c506-40c9-a86a-717d2ef3661a-c000.csv
